In [0]:
# ============================================================
# Project configuration: catalog, schema, raw volume, Auto Loader locations
# ============================================================
catalog_name = "capstone9_smartmeter"
schema_name = "smart_grid_bronze"

base_path = "/Volumes/capstone9_smartmeter/smart_grid_bronze/raw_volume"
schema_base = f"{base_path}/_schemas/smart_grid_new"          # Auto Loader schemaLocation root
checkpoint_base = f"{base_path}/_checkpoints/smart_grid_new"  # Auto Loader checkpointLocation root

# Set the session's default catalog and schema
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

print("Catalog:", catalog_name)
print("Schema:", schema_name)
print("Volume:", base_path)

In [0]:
# Folders are created by Auto Loader on first run (notebook 02)
print("SCHEMA FOLDERS")
display(dbutils.fs.ls(schema_base))

print("CHECKPOINT FOLDERS")
display(dbutils.fs.ls(checkpoint_base))

In [0]:
display(dbutils.fs.ls(base_path))

In [0]:
# Source files per landing folder (uploaded unchanged)
folders = [
    "telemetry_data",
    "event_data",
    "master_data",
    "lookup_data"
]

for folder in folders:

    print(f"\n========== {folder} ==========")

    files = dbutils.fs.ls(f"{base_path}/{folder}")

    for file in files:
        print(file.name, file.path)

In [0]:
# Count all records in CSVs in raw_volume (telemetry_data has subdirectories)
total_records = 0
for folder in ["telemetry_data", "event_data", "master_data", "lookup_data"]:
    path = f"{base_path}/{folder}"
    top_items = dbutils.fs.ls(path)
    csv_files = []
    for item in top_items:
        if item.name.endswith("/"):
            # It's a subdirectory — list its contents for CSVs
            sub_files = dbutils.fs.ls(item.path)
            csv_files.extend([f.path for f in sub_files if f.name.endswith(".csv")])
        elif item.name.endswith(".csv"):
            csv_files.append(item.path)
    if csv_files:
        folder_count = spark.read.option("header", "true").csv(csv_files).count()
    else:
        folder_count = 0
    total_records += folder_count
    print(f"{folder}: {len(csv_files)} CSV files, {folder_count:,} records")

print(f"\nTotal records across all CSVs: {total_records:,}")